In [0]:
# ============================================================
# RETAIL MEDALLION PIPELINE — SILVER LAYER
# Description: Reads Bronze table, applies deduplication,
#              data type casting, and feature engineering.
#              Writes clean, analytics-ready data to Silver.
# ============================================================

BRONZE_TABLE_NAME = "bronze_orders"
SILVER_TABLE_NAME = "silver_orders"

print(" Config loaded.")
print(f"   Source : {BRONZE_TABLE_NAME}")
print(f"   Target : {SILVER_TABLE_NAME}")

 Config loaded.
   Source : bronze_orders
   Target : silver_orders


In [0]:
# ------------------------------------------------------------
# STEP 1: Read from Bronze layer
# ------------------------------------------------------------
# Silver always reads from Bronze — never from raw source.
# This ensures a clean separation of concerns across layers.

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

df_bronze = spark.sql(f"SELECT * FROM {BRONZE_TABLE_NAME}")

print(f" Bronze data loaded. Total records: {df_bronze.count()}")

 Bronze data loaded. Total records: 9994


In [0]:
# ------------------------------------------------------------
# STEP 2: Deduplicate records
# ------------------------------------------------------------
# If the pipeline runs multiple times, duplicate records
# may exist in Bronze. We keep only the latest record
# per order_id using a Window function ranked by
# ingestion_timestamp descending.

from pyspark.sql.functions import row_number, desc
from pyspark.sql.window import Window

window = Window.partitionBy("order_id").orderBy(desc("ingestion_timestamp"))

df_deduped = df_bronze \
    .withColumn("row_num", row_number().over(window)) \
    .filter("row_num = 1") \
    .drop("row_num")

before = df_bronze.count()
after  = df_deduped.count()

print(f" Deduplication complete.")
print(f"   Before : {before} records")
print(f"   After  : {after} records")
print(f"   Removed: {before - after} duplicates")

 Deduplication complete.
   Before : 9994 records
   After  : 9994 records
   Removed: 0 duplicates


In [0]:
# ------------------------------------------------------------
# STEP 3: Cast data types & engineer new features
# ------------------------------------------------------------
# Raw CSV data comes in as strings. We cast to proper types
# and derive business-relevant columns:
#
#   discount_amount    = list_price × (discount_percent / 100)
#   sale_price         = list_price - discount_amount
#   profit             = sale_price - cost_price
#   profit_margin_pct  = (profit / sale_price) × 100
#
# Note: profit_margin_pct is NULL when sale_price = 0
#       to avoid division by zero errors.

from pyspark.sql.functions import (
    to_date, col, round, lit,
    current_timestamp, when
)

df_silver = df_deduped \
    .withColumn("order_date",
        to_date(col("order_date"), "yyyy-MM-dd")) \
    .withColumn("quantity",
        col("quantity").cast("integer")) \
    .withColumn("cost_price",
        col("cost_price").cast("double")) \
    .withColumn("list_price",
        col("list_price").cast("double")) \
    .withColumn("discount_percent",
        col("discount_percent").cast("double")) \
    .withColumn("discount_amount",
        round(col("list_price") * col("discount_percent") / 100, 2)) \
    .withColumn("sale_price",
        round(col("list_price") - col("discount_amount"), 2)) \
    .withColumn("profit",
        round(col("sale_price") - col("cost_price"), 2)) \
    .withColumn("profit_margin_pct",
        when(col("sale_price") == 0, None)
        .otherwise(round((col("profit") / col("sale_price")) * 100, 2))) \
    .withColumn("processed_timestamp", current_timestamp()) \
    .withColumn("pipeline_name", lit("retail_medallion_pipeline")) \
    .drop("ingestion_timestamp", "source_file")

print(" Data types cast and features engineered.")
df_silver.printSchema()

 Data types cast and features engineered.
root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- cost_price: double (nullable = true)
 |-- list_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- pipeline_name: string (nullable = false)
 |-- discount_amount: double (nullable = true)
 |-- sale_price: double (nullable = true)
 |-- profit: double (nullable = true)
 |-- profit_margin_pct: double (nullable = true)
 |-- processed_timestamp: timestamp (nullable = false)



In [0]:
# ------------------------------------------------------------
# STEP 4: Write to Delta Lake as Silver managed table
# ------------------------------------------------------------

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE_NAME)

print(f" Silver Delta table created: {SILVER_TABLE_NAME}")

 Silver Delta table created: silver_orders


In [0]:
# ------------------------------------------------------------
# STEP 5: Verify Silver table
# ------------------------------------------------------------

count = spark.sql(f"SELECT COUNT(*) as total FROM {SILVER_TABLE_NAME}").collect()[0][0]
print(f"✅ Silver table verified. Total records: {count}")

print("\n=== PROFIT SUMMARY BY CATEGORY ===")
spark.sql("""
    SELECT
        category,
        COUNT(*)                        AS total_orders,
        ROUND(SUM(profit), 2)           AS total_profit,
        ROUND(AVG(profit_margin_pct), 2) AS avg_margin_pct
    FROM silver_orders
    GROUP BY category
    ORDER BY total_profit DESC
""").show()

✅ Silver table verified. Total records: 9994

=== PROFIT SUMMARY BY CATEGORY ===
+---------------+------------+------------+--------------+
|       category|total_orders|total_profit|avg_margin_pct|
+---------------+------------+------------+--------------+
|     Technology|        1847|     76433.5|          8.28|
|      Furniture|        2121|     66480.7|          7.32|
|Office Supplies|        6026|     62254.5|          3.56|
+---------------+------------+------------+--------------+

